# Engenharia de Features (v5)
## Histórico de Copas + Valor de Mercado (Modelo Híbrido)

**Novas features em relação à v4:**

| Feature | Fonte | Copas disponíveis |
|---------|-------|------------------|
| `media_gols_ultimas2_copas` | dataset Kaggle | 1994–2022 |
| `fase_ultima_copa` | dataset Kaggle | 1994–2022 |
| `valor_mercado_milhoes` | Transfermarkt (planilha) | 2010–2022 |

**Dois datasets gerados:**
- `features_completo_v4.csv` — 1994–2022 (216 treino) — sem valor de mercado
- `features_completo_v5.csv` — 2010–2022 (96 treino) — com valor de mercado

**Codificação de fase:**
- 0 = não classificou para a Copa anterior
- 1 = fase de grupos
- 2 = oitavas de final
- 3 = quartas de final
- 4 = semifinal
- 5 = final (vice)
- 6 = campeão

## 1. Imports e Carregamento

In [9]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../src/features')
from elo import calcular_elo_historico

df_raw = pd.read_csv('../data/raw/results.csv', parse_dates=['date'])
df_v4  = pd.read_csv('../data/processed/features_completo_v4.csv')

# Valor de mercado histórico
df_mercado = pd.read_excel('../data/raw/football_team_values_2010_2026.xlsx',
                            sheet_name='Football Values')
df_mercado.columns = ['copa', 'selecao', 'valor_mercado']
df_mercado = df_mercado[df_mercado['copa'].isin([2010, 2014, 2018, 2022])]

print(f'Dataset bruto: {df_raw.shape}')
print(f'Dataset v4: {df_v4.shape}')
print(f'Valores de mercado disponíveis: {sorted(df_mercado["copa"].unique())}')
print(f'Total registros mercado: {len(df_mercado)}')

Dataset bruto: (49287, 9)
Dataset v4: (248, 12)
Valores de mercado disponíveis: [2014, 2018, 2022]
Total registros mercado: 96


## 2. Calculando ELO Histórico

In [10]:
df = calcular_elo_historico(df_raw, elo_inicial=1000, k_competitivo=40, k_amistoso=20)
print(f'ELO calculado para {len(df)} jogos')

ELO calculado para 49287 jogos


## 3. Calculando Histórico de Copas por Seleção

Para cada Copa alvo, calculamos o desempenho da seleção nas **duas Copas anteriores**:
- Média de gols marcados por jogo nas últimas 2 Copas
- Fase atingida na Copa imediatamente anterior

In [11]:
copas_datas = {
    1990: ('1990-06-08', '1990-07-08'),
    1994: ('1994-06-17', '1994-07-17'),
    1998: ('1998-06-10', '1998-07-12'),
    2002: ('2002-05-31', '2002-06-30'),
    2006: ('2006-06-09', '2006-07-09'),
    2010: ('2010-06-11', '2010-07-11'),
    2014: ('2014-06-12', '2014-07-13'),
    2018: ('2018-06-14', '2018-07-15'),
    2022: ('2022-11-20', '2022-12-18'),
}

def get_jogos_copa(df, ano):
    inicio, fim = copas_datas[ano]
    return df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= inicio) &
        (df['date'] <= fim)
    ]

def get_fase(df_copa, selecao):
    """Determina a fase atingida pela seleção."""
    jogos = df_copa[
        (df_copa['home_team'] == selecao) |
        (df_copa['away_team'] == selecao)
    ]
    if len(jogos) == 0:
        return 0  # não classificou

    rounds = jogos['tournament'].value_counts()
    # Mapear pelo número de jogos — proxy para a fase
    n = len(jogos)
    if n == 3:   return 1  # fase de grupos
    elif n == 4: return 2  # oitavas
    elif n == 5: return 3  # quartas
    elif n == 6: return 4  # semifinal
    elif n == 7:
        # final ou terceiro lugar — verificar se ganhou o último
        ultimo = jogos.sort_values('date').iloc[-1]
        if ultimo['home_team'] == selecao:
            ganhou = ultimo['home_score'] > ultimo['away_score']
        else:
            ganhou = ultimo['away_score'] > ultimo['home_score']
        return 6 if ganhou else 5
    else:
        return min(n - 2, 6)

def get_media_gols_copa(df_copa, selecao):
    """Calcula média de gols na Copa."""
    jogos = df_copa[
        (df_copa['home_team'] == selecao) |
        (df_copa['away_team'] == selecao)
    ]
    if len(jogos) == 0:
        return np.nan
    gols = []
    for _, row in jogos.iterrows():
        if row['home_team'] == selecao:
            gols.append(row['home_score'])
        else:
            gols.append(row['away_score'])
    return np.mean(gols)

print('Funções de histórico definidas!')

# Teste com o Brasil na Copa 2022
copa22 = get_jogos_copa(df, 2022)
copa18 = get_jogos_copa(df, 2018)
copa14 = get_jogos_copa(df, 2014)
print(f'Brasil fase 2018: {get_fase(copa18, "Brazil")}')
print(f'Brasil fase 2022: {get_fase(copa22, "Brazil")}')
print(f'Brasil gols 2018: {get_media_gols_copa(copa18, "Brazil"):.2f}')
print(f'Brasil gols 2014: {get_media_gols_copa(copa14, "Brazil"):.2f}')

Funções de histórico definidas!
Brasil fase 2018: 3
Brasil fase 2022: 3
Brasil gols 2018: 1.60
Brasil gols 2014: 1.57


## 4. Padronizando Nomes no Dataset de Valor de Mercado

O dataset do Transfermarkt usa alguns nomes diferentes do dataset principal. Precisamos padronizar.

In [12]:
# Mapeamento de nomes Transfermarkt → dataset Kaggle
nome_map = {
    'The Netherlands': 'Netherlands',
    'Türkiye':         'Turkey',
    'Turkiye':         'Turkey',
    'South Korea':     'South Korea',
    'United States':   'United States',
    'Ivory Coast':     'Ivory Coast',
    'Bosnia and Herzegovina': 'Bosnia and Herzegovina',
}

df_mercado['selecao'] = df_mercado['selecao'].replace(nome_map)

print('Seleções no dataset de mercado por Copa:')
for ano in [2010, 2014, 2018, 2022]:
    n = len(df_mercado[df_mercado['copa'] == ano])
    print(f'  {ano}: {n} seleções')

Seleções no dataset de mercado por Copa:
  2010: 0 seleções
  2014: 32 seleções
  2018: 32 seleções
  2022: 32 seleções


## 5. Função de Features v5

In [14]:
def calcular_features_v5(selecao, ciclo, copa_df, copa_ano,
                          df_mercado, copas_datas_dict, min_jogos=15):
    """
    Features v4 + histórico de Copas + valor de mercado.
    Valor de mercado pode ser NaN se não disponível.
    """
    # --- Features v4 base ---
    jogos = ciclo[
        (ciclo['home_team'] == selecao) |
        (ciclo['away_team'] == selecao)
    ].sort_values('date')

    if len(jogos) < min_jogos:
        raise ValueError(f'Apenas {len(jogos)} jogos')

    gm, gs, vit, elo_adv = [], [], [], []
    for _, row in jogos.iterrows():
        if row['home_team'] == selecao:
            gm.append(row['home_score']); gs.append(row['away_score'])
            vit.append(1 if row['home_score'] > row['away_score'] else 0)
            elo_adv.append(row['elo_away_antes'])
        else:
            gm.append(row['away_score']); gs.append(row['home_score'])
            vit.append(1 if row['away_score'] > row['home_score'] else 0)
            elo_adv.append(row['elo_home_antes'])

    gm, gs, vit = np.array(gm), np.array(gs), np.array(vit)
    elo_adv = np.array(elo_adv)

    ult15 = jogos.tail(15)
    gm15, gs15, vit15, ea15 = [], [], [], []
    for _, row in ult15.iterrows():
        if row['home_team'] == selecao:
            gm15.append(row['home_score']); gs15.append(row['away_score'])
            vit15.append(1 if row['home_score'] > row['away_score'] else 0)
            ea15.append(row['elo_away_antes'])
        else:
            gm15.append(row['away_score']); gs15.append(row['home_score'])
            vit15.append(1 if row['away_score'] > row['home_score'] else 0)
            ea15.append(row['elo_home_antes'])

    # Target
    sel_copa = copa_df[
        (copa_df['home_team'] == selecao) |
        (copa_df['away_team'] == selecao)
    ]
    gols_copa = [
        row['home_score'] if row['home_team'] == selecao else row['away_score']
        for _, row in sel_copa.iterrows()
    ]

    # --- Histórico de Copas ---
    anos_copas = sorted(copas_datas_dict.keys())
    idx_atual = anos_copas.index(copa_ano)
    copas_anteriores = anos_copas[max(0, idx_atual-2):idx_atual]

    gols_hist = []
    for ano_hist in copas_anteriores:
        inicio, fim = copas_datas_dict[ano_hist]
        copa_hist = df[
            (df['tournament'] == 'FIFA World Cup') &
            (df['date'] >= inicio) & (df['date'] <= fim)
        ]
        mg = get_media_gols_copa(copa_hist, selecao)
        if not np.isnan(mg):
            gols_hist.append(mg)

    media_gols_hist = np.mean(gols_hist) if gols_hist else 0.0

    # Fase da Copa anterior
    if idx_atual > 0:
        ano_ant = anos_copas[idx_atual - 1]
        inicio_ant, fim_ant = copas_datas_dict[ano_ant]
        copa_ant = df[
            (df['tournament'] == 'FIFA World Cup') &
            (df['date'] >= inicio_ant) & (df['date'] <= fim_ant)
        ]
        fase_ant = get_fase(copa_ant, selecao)
    else:
        fase_ant = 0

    # --- Valor de mercado ---
    row_mercado = df_mercado[
        (df_mercado['copa'] == copa_ano) &
        (df_mercado['selecao'] == selecao)
    ]
    valor_mercado = float(row_mercado['valor_mercado'].values[0]) \
        if len(row_mercado) > 0 else np.nan

    return {
        # v4
        'media_gols_marcados_ciclo': gm.mean(),
        'media_gols_sofridos_ciclo': gs.mean(),
        'pct_vitorias_ciclo':        vit.mean(),
        'total_jogos_ciclo':         len(jogos),
        'media_gols_marcados_ult15': np.array(gm15).mean(),
        'media_gols_sofridos_ult15': np.array(gs15).mean(),
        'pct_vitorias_ult15':        np.array(vit15).mean(),
        'elo_medio_adv_ciclo':       elo_adv.mean(),
        'elo_medio_adv_ult15':       np.array(ea15).mean(),
        # v5 novas
        'media_gols_ultimas2_copas': media_gols_hist,
        'fase_ultima_copa':          fase_ant,
        'valor_mercado_milhoes':     valor_mercado,
        # target
        'media_gols_copa':           np.mean(gols_copa)
    }

print('Função v5 definida!')

Função v5 definida!


## 6. Pipeline Completo — Todas as Copas (1994–2022)

In [15]:
copas = {
    1994: {'ciclo_inicio': '1990-07-09', 'ciclo_fim': '1994-06-16', 'copa_inicio': '1994-06-17', 'copa_fim': '1994-07-17'},
    1998: {'ciclo_inicio': '1994-07-18', 'ciclo_fim': '1998-06-09', 'copa_inicio': '1998-06-10', 'copa_fim': '1998-07-12'},
    2002: {'ciclo_inicio': '1998-07-13', 'ciclo_fim': '2002-05-30', 'copa_inicio': '2002-05-31', 'copa_fim': '2002-06-30'},
    2006: {'ciclo_inicio': '2002-07-01', 'ciclo_fim': '2006-06-08', 'copa_inicio': '2006-06-09', 'copa_fim': '2006-07-09'},
    2010: {'ciclo_inicio': '2006-07-10', 'ciclo_fim': '2010-06-10', 'copa_inicio': '2010-06-11', 'copa_fim': '2010-07-11'},
    2014: {'ciclo_inicio': '2010-07-12', 'ciclo_fim': '2014-06-11', 'copa_inicio': '2014-06-12', 'copa_fim': '2014-07-13'},
    2018: {'ciclo_inicio': '2014-07-14', 'ciclo_fim': '2018-06-13', 'copa_inicio': '2018-06-14', 'copa_fim': '2018-07-15'},
    2022: {'ciclo_inicio': '2018-07-16', 'ciclo_fim': '2022-11-19', 'copa_inicio': '2022-11-20', 'copa_fim': '2022-12-18'},
}

todos_dados = []
filtrados   = []

for ano, datas in copas.items():
    print(f'Processando Copa {ano}...')

    ciclo = df[
        (df['date'] >= datas['ciclo_inicio']) &
        (df['date'] <= datas['ciclo_fim']) &
        (df['tournament'] != 'FIFA World Cup')
    ]
    copa_df = df[
        (df['tournament'] == 'FIFA World Cup') &
        (df['date'] >= datas['copa_inicio']) &
        (df['date'] <= datas['copa_fim'])
    ]

    selecoes = pd.unique(copa_df[['home_team', 'away_team']].values.ravel())

    for selecao in selecoes:
        try:
            resultado = calcular_features_v5(
                selecao, ciclo, copa_df, ano,
                df_mercado, copas_datas, min_jogos=15
            )
            resultado['selecao']   = selecao
            resultado['copa_alvo'] = ano
            todos_dados.append(resultado)
        except ValueError as e:
            filtrados.append({'selecao': selecao, 'copa': ano, 'motivo': str(e)})
        except Exception as e:
            print(f'  Erro em {selecao} ({ano}): {e}')

df_v5 = pd.DataFrame(todos_dados)
print(f'\nDataset v5: {df_v5.shape[0]} linhas × {df_v5.shape[1]} colunas')
print(f'Filtrados: {len(filtrados)}')
print(f'\nValor de mercado disponível por Copa:')
for ano in [1994, 1998, 2002, 2006, 2010, 2014, 2018, 2022]:
    sub = df_v5[df_v5['copa_alvo'] == ano]
    com_mercado = sub['valor_mercado_milhoes'].notna().sum()
    print(f'  {ano}: {com_mercado}/{len(sub)} seleções com valor de mercado')

Processando Copa 1994...
Processando Copa 1998...
Processando Copa 2002...
Processando Copa 2006...
Processando Copa 2010...
Processando Copa 2014...
Processando Copa 2018...
Processando Copa 2022...

Dataset v5: 248 linhas × 15 colunas
Filtrados: 0

Valor de mercado disponível por Copa:
  1994: 0/24 seleções com valor de mercado
  1998: 0/32 seleções com valor de mercado
  2002: 0/32 seleções com valor de mercado
  2006: 0/32 seleções com valor de mercado
  2010: 0/32 seleções com valor de mercado
  2014: 32/32 seleções com valor de mercado
  2018: 32/32 seleções com valor de mercado
  2022: 32/32 seleções com valor de mercado


## 7. Validação — Histórico de Copas

In [16]:
print('Histórico Copa 2022 — exemplos:')
cols = ['selecao', 'copa_alvo', 'media_gols_ultimas2_copas',
        'fase_ultima_copa', 'valor_mercado_milhoes', 'media_gols_copa']
print(df_v5[df_v5['copa_alvo'] == 2022][cols]
      .sort_values('valor_mercado_milhoes', ascending=False)
      .head(10).to_string(index=False))

Histórico Copa 2022 — exemplos:
    selecao  copa_alvo  media_gols_ultimas2_copas  fase_ultima_copa  valor_mercado_milhoes  media_gols_copa
    England       2022                   1.190476                 5                 1499.0         2.600000
     Brazil       2022                   1.585714                 3                 1455.0         1.600000
     France       2022                   2.000000                 6                 1337.0         2.285714
      Spain       2022                   1.541667                 2                 1201.0         2.250000
   Portugal       2022                   1.416667                 2                 1154.0         2.400000
    Germany       2022                   1.619048                 1                 1020.0         2.000000
Netherlands       2022                   2.142857                 0                  756.0         2.000000
  Argentina       2022                   1.321429                 2                  748.0         2.142

## 8. Salvamento

In [17]:
df_v5.to_csv('../data/processed/features_completo_v5.csv', index=False)
print('Dataset v5 salvo em: data/processed/features_completo_v5.csv')
print(f'\nLinhas por Copa:')
print(df_v5['copa_alvo'].value_counts().sort_index())

Dataset v5 salvo em: data/processed/features_completo_v5.csv

Linhas por Copa:
copa_alvo
1994    24
1998    32
2002    32
2006    32
2010    32
2014    32
2018    32
2022    32
Name: count, dtype: int64


## 9. Resumo das Features v5

| Feature | Versão | Disponível em |
|---------|--------|---------------|
| 7 features originais | v1 | 1994–2022 |
| `elo_medio_adv_ciclo/ult15` | v4 | 1994–2022 |
| `media_gols_ultimas2_copas` | **v5** | 1994–2022 |
| `fase_ultima_copa` | **v5** | 1994–2022 |
| `valor_mercado_milhoes` | **v5** | 2010–2022 apenas |

**Próximo passo:** `03_modelos_v5.ipynb` — comparar modelo A (v4, 216 amostras) vs modelo B (v5 híbrido, 96 amostras).